# DNN 훈련 기법 — 배치, 과적합 방지, 최적화

## 목차

| 회차 | 파트 | 주제 | 핵심 내용 |
|------|------|------|----------|
| 1 | Part 1 | 배치와 경사하강법 종류 | BGD / SGD / 미니배치 — 왜 미니배치가 표준인지 |
| 2 | Part 2 | PyTorch DataLoader | Dataset·DataLoader 로 미니배치 자동 생성 |
| 3 | Part 3 | 과적합(Overfitting) | 학습/검증/테스트 분할, 학습 곡선 해석법 |
| 4 | Part 4 | 정규화 (L1, L2) | Weight Decay 가 과적합을 어떻게 막는지 |
| 5 | Part 5 | Dropout | 학습 시 뉴런 무작위 비활성화 |
| 6 | Part 6 | Batch Normalization | 층 입력 분포를 안정화 |
| 7 | Part 7 | Early Stopping + LR Scheduler | 언제 멈출지 + 학습률 스케줄링 |
| 8 | Part 8 | 최적화 알고리즘 비교 | SGD / Momentum / Adam / AdamW |
| 9 | Part 9 | 종합 실습: 손글씨 분류 | sklearn digits 로 위 기법 모두 적용 |

> 이전 챕터(01_DNN_개념)에서 **DNN이 무엇인지·어떻게 동작하는지**를 배웠다면,
> 이번 챕터는 **DNN을 "잘" 학습시키는 기법**을 다룹니다.

In [ ]:
# ============================================================
# 공통 설정 (이 셀을 가장 먼저 실행해주세요!)
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import platform
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정
if platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
elif platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

# PyTorch 재현성
import torch
torch.manual_seed(42)
np.random.seed(42)

print(f"numpy: {np.__version__}")
print(f"torch: {torch.__version__}")
print(f"device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

---

## Part 1: 배치(Batch)와 경사하강법 3가지

### 왜 배치 이야기를 꺼낼까요?

이전 챕터에서는 **전체 데이터**(150개 Iris)를 한번에 넣고 한 걸음 업데이트했습니다.
그런데 MNIST 같이 **6만 장짜리 데이터**를 한번에 넣으면 메모리가 터집니다.
그래서 학습 데이터를 **쪼개서(batch)** 조금씩 학습하는 방법이 표준입니다.

### 3가지 경사하강법

<img src="images/02_dnn_training/batch_comparison.png" width="100%">

| 기법 | 한 번에 사용하는 샘플 수 | 1 epoch 내 업데이트 수 | 특징 |
|------|-------------------------|---------------------|------|
| **BGD** (Batch) | 전체 데이터 | 1번 | 안정적이지만 한 걸음이 느림 |
| **SGD** (Stochastic) | 1개 샘플 | N번 | 빠르지만 경로가 지그재그 |
| **MBGD** (Mini-batch) | 일부(예: 32, 64) | N/B번 | **딥러닝 표준 — 속도와 안정성 균형** |

### 수렴 경로는 실제로 얼마나 다를까요?

간단한 2차원 손실 함수($L(w_1, w_2) = (w_1 - 3)^2 + 10(w_2 - 2)^2$, 계곡 모양)에서
시작점 $(-1, -1)$ 에서 목표점 $(3, 2)$ 까지 이동하는 경로를 비교하면 아래와 같습니다.

<img src="images/02_dnn_training/batch_descent_paths.png" width="100%">

| 관찰 포인트 | 의미 |
|-------------|------|
| **BGD 경로** | 거의 직선에 가까운 매끈한 곡선 — 전체 데이터의 평균 기울기를 쓰므로 방향이 정확합니다 |
| **MBGD 경로** | 약간의 떨림이 있지만 빠르게 수렴 — 일부 데이터만 보지만 평균 방향은 충분히 정확합니다 |
| **SGD 경로** | 큰 진폭으로 지그재그 — 한 샘플만 보므로 기울기 추정이 부정확하지만, 업데이트가 가장 빈번합니다 |

> 💡 SGD 의 흔들림은 **단점이자 장점**입니다. 지그재그가 때로는 **얕은 지역 최소점을 탈출**하는 데 도움이 되기도 합니다.

### 용어 정리

| 용어 | 의미 |
|------|------|
| **배치 크기 (batch size)** | 한 번의 업데이트에 사용하는 샘플 수 |
| **이터레이션 (iteration)** | 1번의 가중치 업데이트 (1 step) |
| **에포크 (epoch)** | 전체 학습 데이터를 한 번 다 훑은 것 |

> 예) 샘플 1,000개 · 배치 크기 100 · 10 에포크
> → 에포크 1당 10 iteration · 총 100 iteration

### 수식

$$w \leftarrow w - \text{lr} \cdot \frac{1}{B}\sum_{i=1}^{B} \frac{\partial L_i}{\partial w}$$

$B$(배치 크기)개 샘플의 기울기 **평균**으로 한 걸음 이동합니다.
$B$가 클수록 평균의 정확도가 높아져 경로가 매끈하지만, 한 번의 업데이트 비용이 커집니다.

### 참고: 로컬 미니멈(Local Minimum) vs 글로벌 미니멈(Global Minimum)

앞서 "SGD의 지그재그가 지역 최소점 탈출에 유리" 라고 언급했습니다. 이 개념을 조금 더 자세히 살펴봅시다.

실제 딥러닝의 손실 함수는 위 예시처럼 단순한 계곡이 아니라, **울퉁불퉁한 산맥** 과 같습니다.
골짜기가 여러 개 있을 때, 우리가 찾고 싶은 것은 **가장 깊은 골짜기** 입니다.

<img src="images/02_dnn_training/local_vs_global_minima.png" width="100%">

| 용어 | 한글 | 의미 |
|------|------|------|
| **Local Minimum** | 로컬 미니멈 / 지역 최소점 | 주변에서만 가장 낮은 점 (진짜 최저점은 아닐 수 있습니다) |
| **Global Minimum** | 글로벌 미니멈 / 전역 최소점 | 손실 함수 **전체** 에서 가장 낮은 점 (우리가 찾고 싶은 답) |
| **Saddle Point** | 안장점 | 한 방향은 내려가지만 다른 방향은 올라가는 점 (안장 모양) |

### 왜 문제가 될까요?

경사하강법은 **현재 위치에서 기울기가 0이 되는 곳**을 찾아 멈춥니다.
그런데 로컬 미니멈에서도 기울기가 0이므로, **거기서 학습이 끝나 버리는 함정** 이 있습니다.

> 😓 글로벌 미니멈을 두고 로컬 미니멈에 갇히면, 모델은 "더 좋은 답이 있는 줄 모른 채" 학습을 마칩니다.

### 로컬 탈출 전략

| 전략 | 원리 |
|------|------|
| **SGD의 노이즈** | 미니배치/샘플마다 기울기가 달라서 **자연스러운 흔들림** 이 생깁니다. 얕은 로컬을 빠져나올 확률이 커집니다 |
| **Momentum / Adam** | 이전 기울기의 **관성** 을 사용해 얕은 웅덩이를 "뛰어넘어" 갑니다 (Part 7~8 에서 다룹니다) |
| **Learning Rate Scheduler** | 초반에는 큰 lr 로 많이 흔들리며 탐색하고, 후반에는 작은 lr 로 정밀 수렴합니다 (Part 7 에서 다룹니다) (Part 7 에서 다룹니다) |
| **좋은 초기값** | 시작점이 글로벌 근처면 갇힐 확률이 낮아집니다 |

> 💡 실전에서 글로벌 미니멈을 정확히 찾는 것은 거의 불가능합니다. 현실적 목표는 **"충분히 좋은 로컬 미니멈"** 을 찾는 것이고, 위 전략들을 조합하면 대부분 잘 동작합니다.

---

## Part 2: PyTorch DataLoader — 미니배치를 자동으로

### DataLoader 란?

매번 **for 문으로 직접 인덱스를 자르기는 번거롭습니다.**
PyTorch는 `Dataset` + `DataLoader` 조합으로 **미니배치·셔플·병렬 로딩**을 자동 처리해줍니다.

<img src="images/02_dnn_training/dataloader_flow.png" width="100%">

### 사용법 3줄 요약

```python
from torch.utils.data import TensorDataset, DataLoader
dataset = TensorDataset(X_tensor, y_tensor)
loader = DataLoader(dataset, batch_size=32, shuffle=True)
for X_batch, y_batch in loader:
    # X_batch.shape = (32, 특성 수)
    ...
```

### 주요 옵션

| 옵션 | 의미 |
|------|------|
| `batch_size` | 배치 크기 (기본 1) |
| `shuffle` | 에포크마다 데이터 섞을지 (학습=True 권장) |
| `num_workers` | 병렬 로딩 워커 수 (0=단일 프로세스) |
| `drop_last` | 마지막 배치가 작을 때 버릴지 |

In [ ]:
# ============================================================
# 실습 2: DataLoader 기본 사용법
# ============================================================
# [시나리오] 1,000명 학생 시험 데이터로 미니배치를 만들어 봅니다.
#   - 입력: (1000, 3) 형태 -- 공부시간, 수면시간, 출석률
#   - 목표: 배치 크기 32 로 자동 분할되는지 확인
import torch
from torch.utils.data import TensorDataset, DataLoader

# 가상 데이터 생성 (1,000명)
n_samples = 1000
X = torch.randn(n_samples, 3)              # 3가지 특성
y = torch.randint(0, 2, (n_samples,))      # 0/1 이진 레이블

# Dataset 생성
dataset = TensorDataset(X, y)
print("=" * 60)
print("Dataset 상태")
print("=" * 60)
print(f"  전체 샘플 수: {len(dataset)}")
print(f"  첫 샘플 -> X: {dataset[0][0].tolist()}, y: {dataset[0][1].item()}")

# DataLoader 래핑
batch_size = 32
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

print(f"\n  배치 크기: {batch_size}")
print(f"  총 배치(이터레이션) 수: {len(loader)}")
print(f"  -> 1000 / 32 = 31.25  ==>  31개 풀배치 + 마지막 8개 배치")

# 첫 3개 배치만 꺼내보기
print(f"\n처음 3개 배치 모양 확인:")
for i, (xb, yb) in enumerate(loader):
    print(f"  배치 {i+1}: X.shape={tuple(xb.shape)}, y.shape={tuple(yb.shape)}, y 앞 5개={yb[:5].tolist()}")
    if i >= 2:
        break

print("\n  해석: 매 배치가 (32, 3) 크기로 자동 분할되었습니다.")
print("        shuffle=True 여서 순서가 섞여 편향을 줄여줍니다.")

In [ ]:
# ============================================================
# 실습 2-B: 붓꽃(Iris) 데이터로 실전 DataLoader 파이프라인
# ============================================================
# [시나리오] 가상 데이터가 아닌 실제 Iris 데이터셋으로 DataLoader 를 구성합니다.
#   - 이전 챕터(01_DNN_개념)에서 사용한 붓꽃 데이터 (150개 샘플, 4개 특성, 3개 품종)
#   - 실전 파이프라인: 데이터 로드 → 분할 → 정규화 → Tensor 변환 → DataLoader
#   - 이 순서가 딥러닝 실무의 표준 흐름입니다
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1단계: 데이터 로드
iris = load_iris()
X, y = iris.data, iris.target
print("=" * 60)
print("1단계: Iris 데이터 로드")
print("=" * 60)
print(f"  전체 샘플: {X.shape[0]}개  |  특성: {X.shape[1]}개 (꽃받침·꽃잎 길이/너비)")
print(f"  클래스: {len(set(y))}개  ->  {list(iris.target_names)}")

# 2단계: 학습/검증 분할 (80:20) -- stratify 로 클래스 균형 유지
X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\n2단계: 학습/검증 분할")
print(f"  학습 {len(X_tr)}개 / 검증 {len(X_val)}개")
print(f"  stratify=y 로 각 품종(0/1/2)이 두 셋에 고르게 분배됨")

# 3단계: 정규화 (학습셋 통계만 사용 -- 데이터 누수 방지)
scaler = StandardScaler()
X_tr = scaler.fit_transform(X_tr)   # 학습셋으로 평균·표준편차 계산
X_val = scaler.transform(X_val)     # 검증은 transform 만 (fit 금지!)
print(f"\n3단계: StandardScaler 로 정규화 (각 특성을 평균 0, 표준편차 1 로)")
print(f"  핵심: scaler.fit 은 학습셋만, 검증/테스트는 transform 만 사용합니다")

# 4단계: Tensor 변환 (float32 입력, long 레이블)
X_tr_t = torch.FloatTensor(X_tr)
y_tr_t = torch.LongTensor(y_tr)
X_val_t = torch.FloatTensor(X_val)
y_val_t = torch.LongTensor(y_val)
print(f"\n4단계: NumPy -> PyTorch Tensor 변환 완료")

# 5단계: Dataset + DataLoader (Part 9 에서도 같은 변수 규칙 사용)
train_dataset_p2 = TensorDataset(X_tr_t, y_tr_t)
train_loader_p2 = DataLoader(train_dataset_p2, batch_size=16, shuffle=True)

val_dataset_p2 = TensorDataset(X_val_t, y_val_t)
val_loader_p2 = DataLoader(val_dataset_p2, batch_size=16, shuffle=False)  # 검증은 shuffle 불필요

print(f"\n5단계: DataLoader 구성 (배치 크기 16)")
print(f"  학습 DataLoader: {len(train_loader_p2)}개 배치 (= ceil({len(X_tr)}/16))")
print(f"  검증 DataLoader: {len(val_loader_p2)}개 배치")

# 첫 배치 살펴보기
first_xb, first_yb = next(iter(train_loader_p2))
print(f"\n첫 배치 확인:")
print(f"  X.shape: {tuple(first_xb.shape)}  (배치 크기 x 특성 수)")
print(f"  y.shape: {tuple(first_yb.shape)}")
print(f"  X 샘플 1개: {[round(v, 3) for v in first_xb[0].tolist()]}")
print(f"  y 앞 8개: {first_yb[:8].tolist()}  (0=setosa, 1=versicolor, 2=virginica)")

print(f"\n관찰 포인트:")
print(f"  - 전처리(분할→정규화→Tensor→DataLoader)가 깔끔히 연결됩니다")
print(f"  - 이 5단계 파이프라인은 Part 9 종합 실습(digits) 에서 그대로 반복 적용됩니다")
print(f"  - 중간 Part(3~8) 는 각 기법에 집중하기 위해 더 작은 합성 데이터를 사용합니다")
print(f"  - shuffle=True(학습) / shuffle=False(검증) 차이도 기억해두세요")

---

## Part 3: 과적합(Overfitting) — 학습 데이터에만 익숙해진 모델

### 과적합이란?

> **학습 데이터**에서는 정확도가 매우 높은데 **처음 보는 데이터**에서는 엉망인 상태입니다.

비유하자면, 기출 문제를 **답만 통째로 외운 학생**이 시험장에 가서 새 문제를 만났을 때 당황하는 상황과 같습니다.

<img src="images/02_dnn_training/overfitting_concept.png" width="100%">

### 왜 생기나요?

| 원인 | 예 |
|------|-----|
| 데이터가 적음 | 10개 샘플로 복잡한 모델 학습 |
| 모델이 너무 복잡 | 은닉 뉴런 1000개로 단순 패턴 학습 |
| 학습을 너무 오래 | 에포크 10,000 돌려 학습 데이터 노이즈까지 외움 |

### 해결 전략 개요

| 방법 | 아이디어 | 다룰 Part |
|------|---------|----------|
| 데이터 분할 (검증셋) | "새 데이터" 역할을 할 셋을 따로 빼두기 | Part 3 (지금) |
| L1/L2 정규화 | 가중치가 커지지 못하게 제약 | Part 4 |
| Dropout | 학습 시 뉴런 일부를 랜덤하게 끄기 | Part 5 |
| Batch Normalization | 각 층 입력 분포를 정규화 | Part 6 |
| Early Stopping | 검증 손실이 오르면 조기 종료 | Part 7 |

### 데이터 분할 3종

| 분할 | 용도 | 비율 예 |
|------|------|---------|
| **학습셋 (train)** | 가중치 업데이트 | 60~80% |
| **검증셋 (valid)** | 하이퍼파라미터 튜닝, 과적합 감지 | 10~20% |
| **테스트셋 (test)** | 최종 성능 보고 (학습 중엔 절대 사용 금지) | 10~20% |

In [ ]:
# ============================================================
# 실습 3: 일부러 과적합 만들기 (작은 데이터 + 큰 노이즈 + 복잡한 모델)
# ============================================================
# [시나리오] sin 곡선 위의 점 10개에 0.3 수준의 노이즈를 섞고,
#   뉴런이 많은 DNN 으로 충분히 학습 -> 학습 손실은 0에 수렴하지만
#   검증(새 데이터) 손실은 훨씬 높은 "과적합" 상태를 관찰합니다.
#   - Part 2 에서 배운 DataLoader 로 학습 루프를 구성합니다.
#   - 데이터가 10개뿐이라 batch_size=10 (사실상 full-batch) 로 설정합니다.
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# 1) 학습/검증 데이터 생성 (Part 4, 7 에서도 이 데이터를 재사용합니다. Part 5 는 digits 데이터 사용)
torch.manual_seed(0)
x_train = torch.linspace(0, 2 * np.pi, 10).unsqueeze(1)
y_train = torch.sin(x_train) + 0.3 * torch.randn_like(x_train)

x_valid = torch.linspace(0, 2 * np.pi, 100).unsqueeze(1)
y_valid = torch.sin(x_valid) + 0.1 * torch.randn_like(x_valid)

# Part 2 에서 배운 DataLoader 로 감싸기 (데이터 10개라 batch_size=10)
train_loader_p3 = DataLoader(TensorDataset(x_train, y_train),
                              batch_size=len(x_train), shuffle=True)

# 2) 일부러 "거대한" 모델 (은닉 뉴런 256개 × 2층)
class BigModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, 1),
        )
    def forward(self, x):
        return self.net(x)

model = BigModel()
optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.MSELoss()

# 3) 학습 + 에포크별 학습/검증 손실 기록
train_losses, valid_losses = [], []
for epoch in range(1, 2001):
    # 학습 (DataLoader 로 미니배치 루프)
    model.train()
    epoch_loss = 0
    for xb, yb in train_loader_p3:
        pred = model(xb)
        loss = criterion(pred, yb)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        epoch_loss += loss.item()
    # 검증
    model.eval()
    with torch.no_grad():
        valid_loss = criterion(model(x_valid), y_valid)
    train_losses.append(epoch_loss / len(train_loader_p3))
    valid_losses.append(valid_loss.item())

print("=" * 60)
print("과적합 유도 학습 결과 (데이터 10개 + 노이즈 0.3 + 모델 256x256)")
print("=" * 60)
print(f"  최종 학습 손실: {train_losses[-1]:.4f}  (거의 0)")
print(f"  최종 검증 손실: {valid_losses[-1]:.4f}")
print(f"  비율(검증/학습): {valid_losses[-1] / max(train_losses[-1], 1e-8):.1f} 배")
print("  -> 학습 데이터는 거의 완벽히 맞추지만 새 데이터에서는 오차가 훨씬 큽니다!")

In [ ]:
# ============================================================
# 실습 3 시각화: 학습/검증 손실 곡선 + 모델 예측 곡선
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), facecolor='#FAFBFC')

# (1) 손실 곡선
ax = axes[0]
ax.set_facecolor('#FAFBFC')
ax.plot(train_losses, color='#3B82F6', linewidth=2, label='학습 손실')
ax.plot(valid_losses, color='#EF4444', linewidth=2, label='검증 손실')
ax.set_xlabel('에포크')
ax.set_ylabel('손실')
ax.set_title('학습 vs 검증 손실 — 학습 손실은 0으로, 검증 손실은 멈춤',
             fontsize=11, fontweight='bold')
ax.legend(loc='upper right')
ax.grid(alpha=0.2)

# (2) 모델 예측 곡선 vs 실제
ax = axes[1]
ax.set_facecolor('#FAFBFC')
model.eval()
with torch.no_grad():
    x_plot = torch.linspace(0, 2 * np.pi, 200).unsqueeze(1)
    y_pred = model(x_plot).squeeze().numpy()

ax.scatter(x_train.squeeze(), y_train.squeeze(), s=70, color='black',
           zorder=3, label='학습 데이터(10개)')
ax.plot(x_plot.squeeze().numpy(), np.sin(x_plot.squeeze().numpy()),
        '--', color='#9CA3AF', linewidth=1.5, label='실제 패턴 sin(x)')
ax.plot(x_plot.squeeze().numpy(), y_pred, color='#EF4444', linewidth=2,
        label='과적합된 모델 예측')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.set_title('모델 예측 곡선 — 학습 점엔 딱 붙지만 과도하게 꿀렁꿀렁',
             fontsize=11, fontweight='bold')
ax.legend(loc='lower left', fontsize=9)
ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

print("관찰 포인트:")
print("  왼쪽 : 학습 손실은 0에 수렴하는 반면 검증 손실은 훨씬 높은 값에 머뭅니다 -> 과적합!")
print("  오른쪽: 모델이 점 10개에 딱 맞추느라 매끈한 sin 곡선을 넘어 꿀렁거립니다.")

---

## Part 4: 정규화(Regularization) — 모델에게 "욕심 내지 마라" 한마디

### 왜 정규화가 과적합을 막나요?

과적합된 모델은 **가중치 값이 지나치게 커지는** 경향이 있습니다.
손실 함수에 "가중치가 크면 벌점" 항을 더해서 가중치를 **작게 유지**하도록 강제합니다.

<img src="images/02_dnn_training/regularization_l1_l2.png" width="100%">

### 수식

**L1 정규화 (Lasso)**
$$L_{\text{total}} = L_{\text{data}} + \lambda \sum_j |w_j|$$

**L2 정규화 (Ridge, Weight Decay)**
$$L_{\text{total}} = L_{\text{data}} + \lambda \sum_j w_j^2$$

| 구분 | L1 | L2 |
|------|-----|-----|
| 가중치 효과 | 일부를 **정확히 0**으로 | 전체적으로 **작게** |
| 결과 모델 | 희소(sparse) | 부드러움(smooth) |
| 딥러닝 주 사용 | 드묾 | **매우 흔함** |

### PyTorch 사용법

```python
optimizer = optim.Adam(model.parameters(), lr=0.01, weight_decay=0.01)
#                                          ^^^^^^^^^^^^^^^^^^^^^ = L2 정규화의 lambda
```

> **주의**: `weight_decay` 가 너무 크면 가중치가 0에 가까워져 **언더피팅(underfit)** 이 됩니다.
> 보통 $10^{-5}$ ~ $10^{-2}$ 범위에서 실험합니다.

In [ ]:
# ============================================================
# 실습 4: L2 정규화 유/무 비교 (Part 3의 과적합 데이터 재사용)
# ============================================================
# [시나리오] Part 3 에서 과적합 났던 상황에 weight_decay 만 추가해서
#   같은 학습을 돌린 뒤 "검증 손실이 얼마나 개선되는가" 를 봅니다.
#   - DataLoader 기반 학습 루프로 Part 3 과 동일한 방식
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
# 앞에서 정의한 BigModel 과 동일하지만, 셀 단독 실행을 위해 다시 정의합니다.
class BigModel2(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, 1),
        )
    def forward(self, x):
        return self.net(x)

# Part 3 의 과적합 데이터에 DataLoader 재적용
train_loader_p4 = DataLoader(TensorDataset(x_train, y_train),
                              batch_size=len(x_train), shuffle=True)

def train_one(weight_decay, epochs=2000):
    torch.manual_seed(0)
    m = BigModel2()
    opt = optim.Adam(m.parameters(), lr=0.01, weight_decay=weight_decay)
    crit = nn.MSELoss()
    tr_ls, va_ls = [], []
    for _ in range(epochs):
        m.train()
        for xb, yb in train_loader_p4:
            pred = m(xb); tl = crit(pred, yb)
            opt.zero_grad(); tl.backward(); opt.step()
        m.eval()
        with torch.no_grad():
            vl = crit(m(x_valid), y_valid)
        tr_ls.append(tl.item()); va_ls.append(vl.item())
    wsum = sum(p.abs().mean().item() for p in m.parameters() if p.requires_grad)
    return tr_ls, va_ls, m, wsum

tr0, va0, m0, wsum0 = train_one(weight_decay=0.00)      # 정규화 OFF
tr1, va1, m1, wsum1 = train_one(weight_decay=0.01)      # 정규화 ON

print("=" * 60)
print("L2 정규화 ON/OFF 비교 (과적합 데이터 10개)")
print("=" * 60)
print(f"  [OFF] weight_decay=0.00  -> 최종 검증 손실 {va0[-1]:.4f}  | 평균 |w| {wsum0:.3f}")
print(f"  [ON ] weight_decay=0.01  -> 최종 검증 손실 {va1[-1]:.4f}  | 평균 |w| {wsum1:.3f}")
print(f"\n  검증 손실 개선율: {(1 - va1[-1]/va0[-1]) * 100:.1f}%")
print(f"  가중치 크기 감소율: {(1 - wsum1/wsum0) * 100:.1f}%")
print("  해석: L2 정규화가 가중치를 작게 유지해 새 데이터 성능을 개선합니다!")

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), facecolor='#FAFBFC')
ax = axes[0]; ax.set_facecolor('#FAFBFC')
ax.plot(va0, color='#EF4444', linewidth=2, label='OFF (weight_decay=0)')
ax.plot(va1, color='#10B981', linewidth=2, label='ON  (weight_decay=0.01)')
ax.set_title('검증 손실 비교'); ax.set_xlabel('에포크'); ax.set_ylabel('검증 손실')
ax.legend(); ax.grid(alpha=0.2)

ax = axes[1]; ax.set_facecolor('#FAFBFC')
with torch.no_grad():
    x_plot = torch.linspace(0, 2*np.pi, 200).unsqueeze(1)
    p0 = m0(x_plot).squeeze().numpy()
    p1 = m1(x_plot).squeeze().numpy()
ax.scatter(x_train.squeeze(), y_train.squeeze(), s=55, color='black', zorder=3, label='데이터')
ax.plot(x_plot.squeeze().numpy(), np.sin(x_plot.squeeze().numpy()),
        '--', color='#9CA3AF', label='실제 sin(x)')
ax.plot(x_plot.squeeze().numpy(), p0, color='#EF4444', linewidth=2, label='OFF (과적합)')
ax.plot(x_plot.squeeze().numpy(), p1, color='#10B981', linewidth=2, label='ON (부드러움)')
ax.set_title('예측 곡선 비교'); ax.legend(fontsize=8); ax.grid(alpha=0.2)
plt.tight_layout(); plt.show()

print("\n관찰 포인트:")
print("  - 초록색(ON)이 빨간색(OFF)보다 검증 손실이 낮고 예측 곡선도 부드럽습니다.")

---

## Part 5: Dropout — 뉴런을 무작위로 끄기

### Dropout 이란?

> 학습 시 **뉴런의 일부를 임의로 0으로 만들어 버립니다.**

매 배치마다 끄는 뉴런이 바뀌기 때문에, 특정 뉴런에만 과하게 의존하지 않도록 **강제 분산 학습**이 됩니다.

<img src="images/02_dnn_training/dropout_concept.png" width="100%">

### 비유

축구팀이 매 경기 **랜덤한 선수 2~3명을 제외**하고 훈련하면,
"나 혼자 믿고 있던" 에이스에 의존하지 못하니 **팀 전체가 고르게** 강해집니다.

### 학습 시 vs 추론 시 — 무엇이 달라지나?

| 상황 | Dropout 동작 | 뉴런 출력 |
|------|--------------|----------|
| **학습 (`model.train()`)** | 확률 p 로 뉴런 OFF, 남은 뉴런은 **스케일 업** | `0` 또는 `a_j / (1-p)` |
| **추론 (`model.eval()`)** | **아무 연산도 하지 않음** (Dropout 자동 해제) | 원래 출력 `a_j` 그대로 |

> ⚠️ 평가 전 반드시 `model.eval()` 을 호출하세요. 안 그러면 매번 예측이 달라집니다!

### 수식 (학습 시)

$$\tilde{a}_j = \frac{m_j}{1-p} \cdot a_j, \quad m_j \sim \text{Bernoulli}(1-p)$$

- `m_j = 0` (확률 p): 해당 뉴런 OFF → 출력 0
- `m_j = 1` (확률 1-p): 해당 뉴런 ON → 출력 `a_j / (1-p)` 로 **증폭**

### 왜 "증폭" 하는가? — 기댓값 보존

`1/(1-p)` 는 **항상 1보다 큰 값**입니다 (p > 0 일 때). 즉 **감쇠가 아니라 증폭(스케일 업)** 입니다.

**수치 예시 (p = 0.5, 원래 출력 `a_j = 3.0` 일 때)**:

| 시나리오 | 출력값 | 평균 |
|----------|--------|------|
| 학습 · ON (확률 50%) | `3.0 × 1/(1-0.5) = 6.0` **(2배 증폭)** | |
| 학습 · OFF (확률 50%) | `0` | |
| **학습 평균** | `0.5 × 6.0 + 0.5 × 0 = 3.0` | **3.0** ✓ |
| **추론** | `3.0` (그대로) | **3.0** ✓ |

> 💡 핵심: 학습 시 남은 뉴런을 증폭해두면 **평균 출력 = 원래 출력** 이 유지됩니다.
> 덕분에 추론 시 아무 연산 없이도 학습과 **같은 스케일** 의 출력이 나옵니다.

### PyTorch 사용법

```python
self.dropout = nn.Dropout(p=0.5)   # 50% 확률로 OFF + 살아남은 뉴런 2배 증폭
```

PyTorch 의 `nn.Dropout` 은 기본적으로 **Inverted Dropout** 방식입니다 — 학습 시 증폭하고, 추론 시에는 자동으로 동작하지 않습니다.

In [ ]:
# ============================================================
# 실습 5: Dropout 유/무 비교 (손글씨 숫자 분류, 과적합 시나리오)
# ============================================================
# [시나리오] sklearn digits (8x8 손글씨 숫자) 중 학습 100장만 사용해 과적합을 유도합니다.
#   - 학습 100장 / 검증 500장
#   - 10 클래스 (숫자 0~9)
#   - 뉴런 256x256 2층 DNN
#   - Dropout p=0.4 를 넣었을 때 검증 손실·정확도가 어떻게 변하는지 비교
#   - DataLoader 로 감싸되, 학습 100장이라 batch_size=100 (full-batch) 로 안정적 비교
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler as _Scaler

_digits = load_digits()
_Xd, _yd = _digits.data, _digits.target

_Xtr, _Xrest, _ytr, _yrest = train_test_split(_Xd, _yd, train_size=100, random_state=42, stratify=_yd)
_Xva, _yva = _Xrest[:500], _yrest[:500]

_scaler_do = _Scaler()
_Xtr = _scaler_do.fit_transform(_Xtr)
_Xva = _scaler_do.transform(_Xva)

X_do_tr = torch.FloatTensor(_Xtr); y_do_tr = torch.LongTensor(_ytr)
X_do_va = torch.FloatTensor(_Xva); y_do_va = torch.LongTensor(_yva)

# DataLoader 로 감싸기 (학습 100 -- full-batch 로 안정적 비교)
do_train_loader = DataLoader(TensorDataset(X_do_tr, y_do_tr),
                              batch_size=len(X_do_tr), shuffle=True)

class DropoutModel(nn.Module):
    def __init__(self, p):
        super().__init__()
        # nn.Dropout(p=0) 은 항등 함수로 동작하므로 if 분기 불필요
        self.net = nn.Sequential(
            nn.Linear(64, 256), nn.ReLU(), nn.Dropout(p),
            nn.Linear(256, 256), nn.ReLU(), nn.Dropout(p),
            nn.Linear(256, 10),
        )
    def forward(self, x):
        return self.net(x)

def train_do(p, epochs=300):
    torch.manual_seed(0)
    m = DropoutModel(p)
    opt = torch.optim.Adam(m.parameters(), lr=0.005)
    crit = nn.CrossEntropyLoss()
    tr_ls, va_ls, va_accs = [], [], []
    for _ in range(epochs):
        m.train()
        for xb, yb in do_train_loader:
            out = m(xb); tl = crit(out, yb)
            opt.zero_grad(); tl.backward(); opt.step()
        m.eval()
        with torch.no_grad():
            vo = m(X_do_va)
            vl = crit(vo, y_do_va).item()
            vacc = (vo.argmax(1) == y_do_va).float().mean().item()
        tr_ls.append(tl.item()); va_ls.append(vl); va_accs.append(vacc)
    return tr_ls, va_ls, va_accs

tr_off, va_off, acc_off = train_do(0.0)
tr_on,  va_on,  acc_on  = train_do(0.4)

va_off_avg = sum(va_off[-50:]) / 50
va_on_avg  = sum(va_on[-50:]) / 50
acc_off_avg = sum(acc_off[-50:]) / 50
acc_on_avg  = sum(acc_on[-50:]) / 50

print("=" * 60)
print("Dropout ON/OFF 비교 (손글씨 digits, 학습 100장으로 과적합 유도)")
print("=" * 60)
print(f"  [OFF] Dropout 없음   -> 검증 손실 {va_off_avg:.4f} | 검증 정확도 {acc_off_avg*100:.1f}%")
print(f"  [ON ] Dropout p=0.4  -> 검증 손실 {va_on_avg:.4f} | 검증 정확도 {acc_on_avg*100:.1f}%")
print(f"\n  검증 손실 감소율: {(1 - va_on_avg/va_off_avg) * 100:.1f}%")
print(f"  검증 정확도 변화: {(acc_on_avg - acc_off_avg)*100:+.1f}%p")
print("  해석: Dropout 이 '과도한 확신' 을 줄여 검증 손실을 크게 낮추고, 정확도도 소폭 올립니다!")

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), facecolor='#FAFBFC')
ax = axes[0]; ax.set_facecolor('#FAFBFC')
ax.plot(va_off, color='#EF4444', linewidth=1.5, alpha=0.85, label='Dropout OFF')
ax.plot(va_on,  color='#10B981', linewidth=1.5, alpha=0.85, label='Dropout ON (p=0.4)')
ax.set_xlabel('에포크'); ax.set_ylabel('검증 손실')
ax.set_title('검증 손실 (낮을수록 좋음)', fontweight='bold')
ax.legend(); ax.grid(alpha=0.2)

ax = axes[1]; ax.set_facecolor('#FAFBFC')
ax.plot([a*100 for a in acc_off], color='#EF4444', linewidth=1.5, alpha=0.85, label='Dropout OFF')
ax.plot([a*100 for a in acc_on],  color='#10B981', linewidth=1.5, alpha=0.85, label='Dropout ON (p=0.4)')
ax.set_xlabel('에포크'); ax.set_ylabel('검증 정확도 (%)')
ax.set_title('검증 정확도 (높을수록 좋음)', fontweight='bold')
ax.legend(); ax.grid(alpha=0.2)
plt.tight_layout(); plt.show()

print("\n관찰 포인트:")
print("  - 왼쪽(검증 손실): Dropout ON(초록)이 OFF(빨강) 대비 훨씬 낮게 유지됩니다.")
print("  - 오른쪽(검증 정확도): Dropout ON 이 안정적이고 OFF 보다 조금 높거나 비슷합니다.")
print("  - OFF 는 학습 데이터에 과도하게 확신을 갖다가 틀릴 때 큰 손실이 발생합니다.")

---

## Part 6: Batch Normalization — 층마다 분포 정리정돈

### BN 이란?

각 층의 출력 분포가 **매 배치마다 들쭉날쭉**하면 다음 층이 학습하기 어렵습니다.
BN은 **현재 배치의 평균과 분산**으로 정규화해서 분포를 **평균 0, 분산 1**에 가깝게 맞춰줍니다.

<img src="images/02_dnn_training/batchnorm_concept.png" width="100%">

### 수식

$$\hat{x}_i = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}, \quad y_i = \gamma \hat{x}_i + \beta$$

| 기호 | 의미 |
|------|------|
| $\mu_B, \sigma_B^2$ | 현재 배치의 평균·분산 |
| $\gamma, \beta$ | 학습되는 스케일·이동 파라미터 |
| $\epsilon$ | 0 나눗셈 방지용 (보통 1e-5) |

### 효과

- 학습 속도 **빨라짐** (더 큰 lr 사용 가능)
- 가중치 **초기값에 덜 민감** (Xavier/He 초기화의 부담을 덜어줍니다)
- 일종의 **정규화 효과** 도 있음
- **깊은 네트워크에서 효과가 극대화** — 10층 이상이면 BN 없이는 학습이 거의 안 될 수도 있습니다

### 참고: 가중치 초기화 (Xavier / He)

BN이 없다면 시작 가중치 분포가 매우 중요합니다. 대표적인 초기화는 아래 2가지입니다.

| 초기화 | 권장 활성화 | 특징 |
|--------|------------|------|
| Xavier (Glorot) | Sigmoid, Tanh | 입력·출력 뉴런 수에 맞춰 분산 조절 |
| He (Kaiming) | ReLU | ReLU에 최적화된 분산 (입력 뉴런 수 기반) |

BN이 있으면 이 부담이 줄지만, 여전히 **초깃값은 He 초기화가 기본값**입니다. PyTorch는 `nn.Linear` 생성 시 자동으로 좋은 초깃값을 세팅해줍니다.

### PyTorch 사용법

```python
self.bn1 = nn.BatchNorm1d(128)   # Linear 출력 채널 수와 동일하게
# 순서: Linear -> BN -> 활성화
```

In [ ]:
# ============================================================
# 실습 6: Batch Normalization 유/무 비교 (Wine 데이터셋, 깊은 DNN)
# ============================================================
# [시나리오] 실제 와인 데이터셋(sklearn.datasets.load_wine)으로 깊은 DNN 을 학습합니다.
#   - 178개 샘플, 13개 특성(알코올·산도·플라보노이드 등), 3종 품종 분류
#   - 깊은 네트워크(8층) + SGD(lr=0.1) 에서 BN 유무가 학습 성공을 가르는지 확인
#   - 모델은 DropoutModel(Part 5) 과 같은 스타일로 단순하게 — BN 플래그 + Dropout 기본 포함
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# --- 1. 데이터 준비 (Part 2-B 와 같은 5단계 파이프라인) ---
wine = load_wine()
X, y = wine.data, wine.target
print("=" * 60)
print("Wine 데이터셋 정보")
print("=" * 60)
print(f"  전체 샘플: {X.shape[0]}개  |  특성: {X.shape[1]}개  |  클래스: {len(set(y))}종")
print(f"  특성 예시: {wine.feature_names[:5]} ...")

X_tr_bn, X_val_bn, y_tr_bn, y_val_bn = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler_bn = StandardScaler()
X_tr_bn = scaler_bn.fit_transform(X_tr_bn)
X_val_bn = scaler_bn.transform(X_val_bn)

X_tr_bn_t = torch.FloatTensor(X_tr_bn); y_tr_bn_t = torch.LongTensor(y_tr_bn)
X_val_bn_t = torch.FloatTensor(X_val_bn); y_val_bn_t = torch.LongTensor(y_val_bn)
loader_bn = DataLoader(TensorDataset(X_tr_bn_t, y_tr_bn_t), batch_size=32, shuffle=True)
print(f"  분할: 학습 {len(X_tr_bn_t)} / 검증 {len(X_val_bn_t)}")

# --- 2. 모델 정의 (DropoutModel 스타일 — 단순 평탄 구조 + Dropout 기본 포함) ---
class DeepDNN(nn.Module):
    def __init__(self, use_bn, p=0.2):
        super().__init__()
        layers = [nn.Linear(13, 64)]
        if use_bn:
            layers.append(nn.BatchNorm1d(64))
        layers += [nn.ReLU(), nn.Dropout(p)]

        # 은닉층 6개 (총 8개 Linear)
        for _ in range(6):
            layers.append(nn.Linear(64, 64))
            if use_bn:
                layers.append(nn.BatchNorm1d(64))
            layers += [nn.ReLU(), nn.Dropout(p)]

        layers.append(nn.Linear(64, 3))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

total_params = sum(p.numel() for p in DeepDNN(use_bn=True).parameters())
print(f"\nDeepDNN(use_bn=True) 파라미터 수: {total_params:,}개")

# --- 3. BN ON/OFF 각각 학습 ---
def train_bn(use_bn, epochs=25, lr=0.1):
    torch.manual_seed(0)
    model_bn = DeepDNN(use_bn=use_bn)
    opt = optim.SGD(model_bn.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    losses = []
    for _ in range(epochs):
        model_bn.train()
        ep_loss = 0
        for xb, yb in loader_bn:
            out = model_bn(xb); loss = crit(out, yb)
            opt.zero_grad(); loss.backward(); opt.step()
            ep_loss += loss.item()
        losses.append(ep_loss / len(loader_bn))
    model_bn.eval()
    with torch.no_grad():
        val_acc = (model_bn(X_val_bn_t).argmax(1) == y_val_bn_t).float().mean().item()
    return losses, val_acc

loss_no_bn, acc_no_bn = train_bn(use_bn=False)
loss_bn,    acc_bn    = train_bn(use_bn=True)

# --- 4. 결과 출력 ---
print("\n" + "=" * 60)
print("BatchNorm ON/OFF 에포크별 학습 손실 (Wine + 8층 DNN + SGD lr=0.1)")
print("=" * 60)
print(f"{'에포크':>6} | {'BN OFF':>10} | {'BN ON':>10}")
print("-" * 36)
for i, (a, b) in enumerate(zip(loss_no_bn, loss_bn), 1):
    print(f"{i:>6} | {a:>10.4f} | {b:>10.4f}")

print(f"\n최종 검증 정확도 비교:")
print(f"  [OFF] BN 없음  -> {acc_no_bn*100:.1f}%  (거의 랜덤 추측 수준)")
print(f"  [ON ] BN 적용  -> {acc_bn*100:.1f}%  (거의 완벽 분류)")

# --- 5. 시각화 ---
fig, ax = plt.subplots(figsize=(10, 4.8), facecolor='#FAFBFC')
ax.set_facecolor('#FAFBFC')
ax.plot(loss_no_bn, color='#EF4444', linewidth=2.2, marker='o', label=f'BN OFF (정확도 {acc_no_bn*100:.0f}%)')
ax.plot(loss_bn,    color='#10B981', linewidth=2.2, marker='o', label=f'BN ON  (정확도 {acc_bn*100:.0f}%)')
ax.set_xlabel('에포크'); ax.set_ylabel('평균 학습 손실')
ax.set_title('Batch Normalization 유/무에 따른 수렴 (Wine · 8층 깊은 DNN)', fontweight='bold')
ax.legend(fontsize=11); ax.grid(alpha=0.2)
plt.tight_layout(); plt.show()

print("\n관찰 포인트:")
print(f"  - BN OFF(빨강): 손실이 ~{loss_no_bn[0]:.2f} 에서 거의 움직이지 않습니다. 8층의 기울기 흐름이 막혀 학습 실패!")
print(f"  - BN ON(초록) : 같은 조건에서 손실이 꾸준히 떨어져 검증 정확도 {acc_bn*100:.0f}% 까지 올라갑니다.")
print( "  - 얕은 네트워크(2~3층)에서는 차이가 작지만, 깊을수록 BN 의 가치가 커집니다.")
print( "  - BN 이 '깊은 네트워크를 학습 가능하게' 만들어준다는 핵심 메시지!")

---

## Part 7: Early Stopping & Learning Rate Scheduler

### 두 가지 "학습 제어" 기법

<img src="images/02_dnn_training/early_stopping_lr_schedule.png" width="100%">

### 1) Early Stopping — 언제 멈출지

검증 손실이 **일정 에포크 동안 개선되지 않으면** 학습을 조기 종료합니다.

```
아이디어:
  best_loss, patience_counter = inf, 0
  for epoch in ...:
      valid_loss = evaluate()
      if valid_loss < best_loss:
          best_loss = valid_loss
          patience_counter = 0       # 개선됐으니 리셋
          save_best_weights()
      else:
          patience_counter += 1
          if patience_counter >= patience:
              break                  # 더 이상 개선 없음 -> 종료!
```

| 하이퍼파라미터 | 의미 | 일반 값 |
|---------------|------|--------|
| `patience` | 몇 에포크까지 참고 기다릴지 | 5 ~ 20 |
| `min_delta` | 개선으로 인정할 최소 차이 | 0.0 ~ 1e-4 |

### 2) LR Scheduler — 학습률을 조절

학습 초반엔 큰 lr로 빠르게 내려가다가, 후반엔 작은 lr로 **세밀하게** 수렴합니다.

| Scheduler | 동작 |
|-----------|------|
| `StepLR(step_size=30, gamma=0.5)` | 30 에포크마다 lr을 절반으로 |
| `ReduceLROnPlateau(...)` | 검증 손실이 정체되면 자동으로 lr 감소 |
| `CosineAnnealingLR(T_max)` | lr 을 코사인 곡선으로 감소 |

### PyTorch 사용법

```python
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.5)
for epoch in ...:
    train_one_epoch()
    scheduler.step()      # 에포크 끝에 호출
```

In [ ]:
# ============================================================
# 실습 7: Early Stopping + LR Scheduler (Wine 데이터셋, 실무 시나리오)
# ============================================================
# [시나리오] Part 6 에서 소개한 Wine 데이터셋(13특성·3클래스)으로 실제 학습 제어 기법을 시연합니다.
#   - BN + Dropout 포함 얕은 DNN 을 Adam 으로 학습
#   - StepLR: 10 에포크마다 lr 을 절반으로 (실무에서 가장 흔한 패턴)
#   - Early Stopping: patience=7 (검증 손실이 7 에포크 개선 없으면 종료)
#   - 학습 최대치는 100 에포크로 여유있게 두고, 실제로는 조기 종료되는 모습 관찰
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from copy import deepcopy

# --- 1. 데이터 준비 (Part 2-B 와 같은 5단계 파이프라인) ---
wine = load_wine()
X, y = wine.data, wine.target
X_tr_es, X_val_es, y_tr_es, y_val_es = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
scaler_es = StandardScaler()
X_tr_es = scaler_es.fit_transform(X_tr_es)
X_val_es = scaler_es.transform(X_val_es)

X_tr_es_t = torch.FloatTensor(X_tr_es); y_tr_es_t = torch.LongTensor(y_tr_es)
X_val_es_t = torch.FloatTensor(X_val_es); y_val_es_t = torch.LongTensor(y_val_es)
loader_es = DataLoader(TensorDataset(X_tr_es_t, y_tr_es_t), batch_size=16, shuffle=True)
print(f"Wine 분할: 학습 {len(X_tr_es_t)} / 검증 {len(X_val_es_t)}")

# --- 2. 모델 (Part 5 Dropout + Part 6 BN 를 결합한 실무형 DNN) ---
class WineNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(13, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(32, 3),
        )
    def forward(self, x): return self.net(x)

torch.manual_seed(0)
model = WineNet()
optimizer = optim.Adam(model.parameters(), lr=0.01)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)  # 실무 표준
criterion = nn.CrossEntropyLoss()

# --- 3. Early Stopping 구성 ---
patience = 7
best_valid = float('inf')
best_epoch = 0
best_weights = None
wait = 0

# --- 4. 학습 루프 (최대 100 에포크, 조기 종료 감지) ---
lr_log, train_log, valid_log = [], [], []
stopped_at = None
MAX_EPOCHS = 100

for epoch in range(1, MAX_EPOCHS + 1):
    # 학습
    model.train()
    ep_tr_loss, n_b = 0, 0
    for xb, yb in loader_es:
        out = model(xb); tl = criterion(out, yb)
        optimizer.zero_grad(); tl.backward(); optimizer.step()
        ep_tr_loss += tl.item(); n_b += 1
    tr_avg = ep_tr_loss / n_b

    # 검증
    model.eval()
    with torch.no_grad():
        vl = criterion(model(X_val_es_t), y_val_es_t).item()

    train_log.append(tr_avg); valid_log.append(vl)
    lr_log.append(optimizer.param_groups[0]['lr'])
    scheduler.step()

    # Early Stopping 판정
    if vl < best_valid - 1e-4:
        best_valid = vl; best_epoch = epoch
        best_weights = deepcopy(model.state_dict())
        wait = 0
    else:
        wait += 1
        if wait >= patience:
            stopped_at = epoch
            break

# 최적 가중치 복원
model.load_state_dict(best_weights)
model.eval()
with torch.no_grad():
    val_acc = (model(X_val_es_t).argmax(1) == y_val_es_t).float().mean().item()

# --- 5. 결과 출력 ---
print("\n" + "=" * 60)
print("Early Stopping + StepLR(10 에포크마다 절반) 학습 결과")
print("=" * 60)
print(f"  최대 에포크 설정   : {MAX_EPOCHS}")
print(f"  실제 조기 종료     : {stopped_at if stopped_at else '끝까지 학습'} 에포크")
print(f"  최적 에포크        : {best_epoch} (이때 검증 손실 {best_valid:.4f})")
print(f"  최적 가중치 복원 후 검증 정확도: {val_acc*100:.1f}%")
print(f"  학습률 변화        : {lr_log[0]:.4f} -> {lr_log[-1]:.5f}")

# lr 감소 지점 탐지
lr_changes = []
for i in range(1, len(lr_log)):
    if lr_log[i] != lr_log[i-1]:
        lr_changes.append(i+1)
print(f"  lr 감소 지점       : {lr_changes} 에포크")

# --- 6. 시각화 ---
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), facecolor='#FAFBFC')
ax = axes[0]; ax.set_facecolor('#FAFBFC')
ax.plot(range(1, len(train_log)+1), train_log, color='#3B82F6', linewidth=2, label='학습 손실')
ax.plot(range(1, len(valid_log)+1), valid_log, color='#EF4444', linewidth=2, label='검증 손실')
ax.axvline(best_epoch, color='#10B981', linestyle='--', linewidth=2,
           label=f'최적 에포크 ({best_epoch})')
if stopped_at:
    ax.axvline(stopped_at, color='#9CA3AF', linestyle=':', linewidth=1.5,
               label=f'조기 종료 ({stopped_at})')
ax.set_xlabel('에포크'); ax.set_ylabel('손실')
ax.set_title('Early Stopping 시점', fontweight='bold')
ax.legend(); ax.grid(alpha=0.2)

ax = axes[1]; ax.set_facecolor('#FAFBFC')
ax.plot(range(1, len(lr_log)+1), lr_log, color='#8B5CF6', linewidth=2.2, marker='o', markersize=3)
ax.set_xlabel('에포크'); ax.set_ylabel('학습률 (lr)')
ax.set_title('StepLR: 10 에포크마다 절반', fontweight='bold')
ax.grid(alpha=0.2)
plt.tight_layout(); plt.show()

print("\n관찰 포인트:")
print(f"  왼쪽 (손실) : 초록 점선이 '검증 손실 최저' 지점. 이후 {patience}에포크 참다가 회색 점선에서 종료.")
print( "              학습 손실(파랑)과 검증 손실(빨강) 모두 초반에 급락 후 안정됩니다.")
print( "  오른쪽 (lr): 10 에포크마다 절반으로 줄어드는 계단형 — 실무에서 가장 흔한 스케줄입니다.")
print(f"  -> 100 에포크 설정했지만 Early Stopping 덕에 {stopped_at if stopped_at else '-'} 에포크에서 자동 종료!")

---

## Part 8: 옵티마이저 비교 — SGD / Momentum / Adam / AdamW

<img src="images/02_dnn_training/optimizer_comparison.png" width="100%">

### 한눈에 보기

| 옵티마이저 | 수식 요약 | 실무 특징 |
|-----------|----------|----------|
| **SGD** | $w \leftarrow w - \text{lr} \cdot g$ | 단순·일반화 잘 됨, 수렴 느림 |
| **Momentum** | $v \leftarrow \beta v + g, \, w \leftarrow w - \text{lr} \cdot v$ | 관성 이용, 지그재그 감소 |
| **Adam** | Momentum + 파라미터별 lr 적응 | 빠른 수렴, 간단한 실험용 |
| **AdamW** | Adam + **올바른 weight decay** | **2025년 실무·최신 모델의 표준** |

> 💡 **AdamW란?** Adam의 weight decay 구현이 수학적으로 잘못되어 있었는데, 이를 고친 버전입니다 (Loshchilov & Hutter, 2017). 요즘 대부분의 논문·프로덕션은 `AdamW`를 기본값으로 사용합니다.

### 언제 무엇을 쓰나요? (2025년 기준)

| 상황 | 추천 | 이유 |
|------|------|------|
| 처음 시작 / 빠른 실험 | **Adam** (lr=0.001) | 설정이 간단하고 웬만하면 수렴함 |
| **대부분의 실무·대회** | **AdamW** (lr=0.001, weight_decay=0.01) | 일반화·안정성 모두 우수한 현대 표준 |
| **Transformer / LLM / 확산모델** | **AdamW** (사실상 필수) | BERT·GPT·Llama·Stable Diffusion 모두 AdamW |
| **Vision Transformer / ConvNeXt** | **AdamW** (lr=1e-4, weight_decay=0.05) | 최신 vision 모델도 AdamW로 전환 |
| ResNet 등 고전 CNN 재현 | SGD + Momentum (lr=0.1, momentum=0.9) | ImageNet 레시피 재현 시 여전히 사용 |

### PyTorch 사용법

```python
# 2025 실무 기본값 - 웬만하면 이거로 시작
optim.AdamW(params, lr=0.001, weight_decay=0.01)

# 빠른 실험 / 간단한 모델
optim.Adam(params, lr=0.001)

# 고전 CNN 베이스라인 재현
optim.SGD(params, lr=0.1, momentum=0.9, weight_decay=1e-4)
```

> ⚠️ 옛날 자료에서 "Adam으로 학습 후 SGD로 파인튜닝하면 일반화가 좋아진다"는 레시피를 볼 수 있는데, 이는 주로 2018~2019년 ImageNet CNN 시절의 이야기입니다. 최근에는 **AdamW 단독**으로도 충분한 일반화 성능이 나옵니다.

In [ ]:
# ============================================================
# 실습 8: 3가지 옵티마이저 동시 학습 비교
# ============================================================
# [시나리오] 동일한 모델·동일한 데이터로 SGD / Momentum / Adam 을 돌려
#   학습 곡선이 어떻게 다른지 관찰합니다.
#   - 1,500개 데이터 + DataLoader (batch_size=64, 실제 미니배치 훈련)
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(1)
N_opt, D_opt = 1500, 10
X_opt = torch.randn(N_opt, D_opt)
w_opt = torch.randn(D_opt, 1)
y_opt = (X_opt @ w_opt + 0.3 * torch.randn(N_opt, 1) > 0).long().squeeze()

opt_loader = DataLoader(TensorDataset(X_opt, y_opt), batch_size=64, shuffle=True)

def build_small():
    return nn.Sequential(nn.Linear(D_opt, 32), nn.ReLU(),
                         nn.Linear(32, 32), nn.ReLU(),
                         nn.Linear(32, 2))

def train_opt(opt_name, epochs=30, lr=0.01):
    torch.manual_seed(0)
    m = build_small()
    if opt_name == 'SGD':
        opt = torch.optim.SGD(m.parameters(), lr=lr)
    elif opt_name == 'Momentum':
        opt = torch.optim.SGD(m.parameters(), lr=lr, momentum=0.9)
    elif opt_name == 'Adam':
        opt = torch.optim.Adam(m.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    losses = []
    for _ in range(epochs):
        m.train()
        ep_loss, n_b = 0, 0
        for xb, yb in opt_loader:
            out = m(xb); loss = crit(out, yb)
            opt.zero_grad(); loss.backward(); opt.step()
            ep_loss += loss.item(); n_b += 1
        losses.append(ep_loss / n_b)
    return losses

curves = {name: train_opt(name) for name in ['SGD', 'Momentum', 'Adam']}

print("=" * 60)
print("옵티마이저별 에포크당 평균 학습 손실 (30 에포크 · batch_size=64)")
print("=" * 60)
for name, ls in curves.items():
    print(f"  {name:>10}: 초기 {ls[0]:.4f}  ->  최종 {ls[-1]:.4f}")

fig, ax = plt.subplots(figsize=(10, 5), facecolor='#FAFBFC')
ax.set_facecolor('#FAFBFC')
colors = {'SGD': '#3B82F6', 'Momentum': '#8B5CF6', 'Adam': '#14B8A6'}
for name, ls in curves.items():
    ax.plot(ls, color=colors[name], linewidth=2.2, marker='o', markersize=4, label=name)
ax.set_xlabel('에포크'); ax.set_ylabel('평균 학습 손실')
ax.set_title('옵티마이저 비교: SGD vs Momentum vs Adam (DataLoader 미니배치)', fontweight='bold')
ax.legend(); ax.grid(alpha=0.2)
plt.tight_layout(); plt.show()

print("\n관찰 포인트:")
print("  - SGD     : 완만히 내려가고 수렴이 늦습니다.")
print("  - Momentum: 관성 덕분에 SGD 보다 빠릅니다.")
print("  - Adam    : 초반부터 빠르게 내려가 가장 빨리 수렴합니다 -> 실무 기본값인 이유!")

---

## Part 9: 종합 실습 — 손글씨 숫자 분류

### 데이터셋: sklearn digits (축소판 MNIST)

진짜 MNIST 는 28x28 · 70,000장이지만, 이번 실습에서는 **강의실 환경에서 즉시 실행 가능**하도록
sklearn 에 내장된 **`load_digits`** (축소판 MNIST) 를 사용합니다.

| 항목 | 값 |
|------|-----|
| 이미지 크기 | 8 x 8 (64 픽셀, 그레이스케일) |
| 클래스 | 10개 (숫자 0~9) |
| 전체 데이터 | **1,797장** |
| 분할 (이번 실습) | 학습 1,078 / 검증 359 / 테스트 360 |

> 원본 MNIST(28×28) 에서도 동일한 코드 구조가 작동합니다. 픽셀 수(`64`)만 `784` 로 바꾸면 됩니다.

### 적용할 기법 종합 체크리스트

| 기법 | 어디서 |
|------|--------|
| DataLoader + 미니배치 | `batch_size=64, shuffle=True` |
| Dropout | 은닉층 뒤에 `nn.Dropout(0.3)` |
| BatchNorm | 각 Linear 뒤 `nn.BatchNorm1d(...)` |
| 옵티마이저 AdamW | `optim.AdamW(..., weight_decay=1e-4)` (Part 8 에서 배운 실무 표준) |
| LR Scheduler | `StepLR(step_size=5, gamma=0.5)` |
| Early Stopping | 검증 손실 기반 patience=5 |
| 검증셋 분리 | 학습 데이터 중 20% |

### 비유

손글씨 분류기는 **우체국의 우편번호 자동 인식기** 의 축소판입니다. 오늘 배운 기법을 모두 동원해서 "튼튼한" 인식기를 만들어 봅니다.

In [ ]:
# ============================================================
# 실습 9-1: 데이터 준비 (sklearn digits)
# ============================================================
# [시나리오] sklearn.datasets.load_digits 사용
#   - 8x8 (64 픽셀) 그레이스케일 손글씨 숫자 데이터
#   - 10 클래스 (숫자 0~9)
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

digits = load_digits()
X, y = digits.data, digits.target   # X: (1797, 64), y: (1797,)

print("=" * 60)
print("Digits 데이터셋 정보")
print("=" * 60)
print(f"  전체 이미지: {X.shape[0]}장, 픽셀 수: {X.shape[1]} (=8x8)")
print(f"  클래스 수: {len(np.unique(y))}개  (0~9)")

# train/valid/test = 60 / 20 / 20
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)
X_val, X_te, y_val, y_te = train_test_split(X_tmp, y_tmp, test_size=0.5, random_state=42, stratify=y_tmp)

# 정규화
scaler = StandardScaler()
X_tr = scaler.fit_transform(X_tr)
X_val = scaler.transform(X_val)
X_te = scaler.transform(X_te)

print(f"  분할 결과 -> 학습 {X_tr.shape[0]} / 검증 {X_val.shape[0]} / 테스트 {X_te.shape[0]}")

# 샘플 이미지 몇 장 시각화
fig, axes = plt.subplots(2, 5, figsize=(10, 4.2), facecolor='#FAFBFC')
for ax, img, label in zip(axes.flat, digits.images[:10], digits.target[:10]):
    ax.imshow(img, cmap='gray_r')
    ax.set_title(f'Label: {label}', fontsize=10)
    ax.axis('off')
plt.suptitle('Digits 샘플 10장', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# 실습 9-2: DNN 모델 정의 (Dropout + BatchNorm 포함)
# ============================================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# 텐서로 변환
X_tr_t = torch.FloatTensor(X_tr);   y_tr_t = torch.LongTensor(y_tr)
X_val_t = torch.FloatTensor(X_val); y_val_t = torch.LongTensor(y_val)
X_te_t = torch.FloatTensor(X_te);   y_te_t = torch.LongTensor(y_te)

# DataLoader
train_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=64, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=64)
test_loader  = DataLoader(TensorDataset(X_te_t, y_te_t), batch_size=64)

class DigitsNet(nn.Module):
    def __init__(self):
        super().__init__()
        # 64 -> 128(BN+ReLU+Dropout) -> 64(BN+ReLU+Dropout) -> 10
        self.fc1 = nn.Linear(64, 128)
        self.bn1 = nn.BatchNorm1d(128)
        self.fc2 = nn.Linear(128, 64)
        self.bn2 = nn.BatchNorm1d(64)
        self.fc3 = nn.Linear(64, 10)
        self.dropout = nn.Dropout(p=0.3)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.bn1(self.fc1(x)))
        x = self.dropout(x)
        x = self.relu(self.bn2(self.fc2(x)))
        x = self.dropout(x)
        x = self.fc3(x)   # 출력층 (CrossEntropyLoss 가 내부 Softmax)
        return x

torch.manual_seed(42)
model = DigitsNet()
total_params = sum(p.numel() for p in model.parameters())
print("=" * 60)
print("DigitsNet 구조")
print("=" * 60)
print(model)
print(f"\n총 파라미터 수: {total_params:,}개")

In [ ]:
# ============================================================
# 실습 9-3: 학습 루프 (Adam + weight_decay + StepLR + Early Stopping)
# ============================================================
from copy import deepcopy

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.005, weight_decay=1e-4)  # AdamW + L2 포함 (Part 8 참고)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

def evaluate(loader):
    model.eval()
    tot_loss, correct, n = 0, 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            out = model(xb)
            tot_loss += criterion(out, yb).item() * xb.size(0)
            correct += (out.argmax(1) == yb).sum().item()
            n += xb.size(0)
    return tot_loss / n, correct / n

# Early Stopping 설정
patience = 5
best_vl = float('inf'); best_state = None; wait = 0; stopped = None

train_losses, valid_losses, valid_accs, lr_history = [], [], [], []

for epoch in range(1, 51):
    # 학습
    model.train()
    ep_loss, n = 0, 0
    for xb, yb in train_loader:
        out = model(xb); loss = criterion(out, yb)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        ep_loss += loss.item() * xb.size(0); n += xb.size(0)
    tr_loss = ep_loss / n
    # 검증
    vl_loss, vl_acc = evaluate(val_loader)
    train_losses.append(tr_loss); valid_losses.append(vl_loss); valid_accs.append(vl_acc)
    lr_history.append(optimizer.param_groups[0]['lr'])
    scheduler.step()

    if epoch % 5 == 0 or epoch == 1:
        print(f"  Epoch {epoch:2d} | lr={lr_history[-1]:.4f} | 학습손실={tr_loss:.4f} "
              f"| 검증손실={vl_loss:.4f} | 검증정확도={vl_acc*100:.1f}%")

    # Early Stopping
    if vl_loss < best_vl - 1e-5:
        best_vl = vl_loss; best_state = deepcopy(model.state_dict()); wait = 0
    else:
        wait += 1
        if wait >= patience:
            stopped = epoch
            break

print("\n" + "=" * 60)
print(f"학습 종료: 총 {len(train_losses)} 에포크" +
      (f" (Early Stopping @ {stopped})" if stopped else ""))
print("=" * 60)

# 최적 가중치 복원 후 테스트 성능 보고
model.load_state_dict(best_state)
te_loss, te_acc = evaluate(test_loader)
print(f"  최종 테스트 손실   : {te_loss:.4f}")
print(f"  최종 테스트 정확도 : {te_acc*100:.2f}%")

In [ ]:
# ============================================================
# 실습 9-4: 학습 과정 시각화
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), facecolor='#FAFBFC')

# (1) 손실 곡선
ax = axes[0]; ax.set_facecolor('#FAFBFC')
ax.plot(train_losses, color='#3B82F6', linewidth=2, label='학습 손실')
ax.plot(valid_losses, color='#EF4444', linewidth=2, label='검증 손실')
ax.set_xlabel('에포크'); ax.set_ylabel('손실')
ax.set_title('학습/검증 손실'); ax.legend(); ax.grid(alpha=0.2)

# (2) 검증 정확도
ax = axes[1]; ax.set_facecolor('#FAFBFC')
ax.plot([a * 100 for a in valid_accs], color='#10B981', linewidth=2, marker='o', markersize=3)
ax.set_xlabel('에포크'); ax.set_ylabel('검증 정확도 (%)')
ax.set_title('검증 정확도'); ax.grid(alpha=0.2)

# (3) 학습률 스케줄
ax = axes[2]; ax.set_facecolor('#FAFBFC')
ax.plot(lr_history, color='#8B5CF6', linewidth=2)
ax.set_xlabel('에포크'); ax.set_ylabel('학습률 (lr)')
ax.set_title('StepLR 스케줄 (5 에포크마다 절반)'); ax.grid(alpha=0.2)

plt.tight_layout(); plt.show()

# 테스트 샘플 몇 개 예측 시각화
model.eval()
with torch.no_grad():
    sample_idx = np.random.choice(len(X_te), 10, replace=False)
    sample_X = torch.FloatTensor(X_te[sample_idx])
    sample_preds = model(sample_X).argmax(1).numpy()

fig, axes = plt.subplots(2, 5, figsize=(10, 4.5), facecolor='#FAFBFC')
for ax, idx, pred in zip(axes.flat, sample_idx, sample_preds):
    img = scaler.inverse_transform(X_te[idx:idx+1]).reshape(8, 8)
    true = y_te[idx]
    ok = pred == true
    ax.imshow(img, cmap='gray_r')
    color = '#10B981' if ok else '#EF4444'
    mark = 'O' if ok else 'X'
    ax.set_title(f'[{mark}] 정답={true} 예측={pred}', fontsize=9, color=color)
    ax.axis('off')
plt.suptitle('테스트 샘플 10장 예측 결과', fontweight='bold')
plt.tight_layout(); plt.show()

print("관찰 포인트:")
print("  - 학습/검증 손실이 같이 내려가고 검증 정확도가 높아집니다.")
print("  - lr 이 계단식으로 감소하면서 후반 학습이 안정됩니다.")
print("  - 숫자 샘플 10개 중 대부분이 초록색(정답)으로 표시됩니다.")

---

## 핵심 정리

### DNN 훈련 기법 한눈에 보기

| 주제 | 핵심 |
|------|------|
| **미니배치 GD** | 데이터를 32~256 씩 쪼개 업데이트 — 속도와 안정성의 균형 |
| **DataLoader** | `TensorDataset` + `DataLoader` 로 배치·셔플 자동화 |
| **과적합** | 학습은 잘 되는데 새 데이터에서 실패하는 현상 |
| **L2 정규화** | 가중치 제곱합을 손실에 더해 가중치를 작게 유지 (`weight_decay`) |
| **Dropout** | 학습 시 뉴런 일부를 무작위로 끕니다 — 추론 시에는 자동 OFF |
| **Batch Normalization** | 각 층 입력을 평균 0, 분산 1로 정규화 — 깊은 네트워크를 살립니다 |
| **Early Stopping** | 검증 손실이 멈추면 조기 종료 — patience 파라미터 중요 |
| **LR Scheduler** | 학습 중 학습률을 점진적으로 낮춰 미세 조정 |
| **Adam / AdamW** | 실무 기본값 옵티마이저 — 빠른 수렴 (AdamW 가 2025 표준) |

### 핵심 코드 정리

| 코드 | 역할 |
|------|------|
| `TensorDataset(X, y)` | 입력·레이블 묶기 |
| `DataLoader(dataset, batch_size=..., shuffle=True)` | 미니배치 자동 생성 |
| `nn.Dropout(p=0.3)` | Dropout 층 (학습 시 30% OFF) |
| `nn.BatchNorm1d(n)` | 1D Batch Normalization |
| `optim.AdamW(..., weight_decay=1e-4)` | AdamW — Adam + 올바른 L2 정규화 (실무 표준) |
| `optim.lr_scheduler.StepLR(...)` | 계단식 학습률 감소 |
| `scheduler.step()` | 에포크 끝에 학습률 갱신 |
| `model.train()` / `model.eval()` | Dropout·BN 모드 전환 |
| `deepcopy(model.state_dict())` | Early Stopping 용 최적 가중치 저장 |

### DNN의 한계 — 다음 시간 예고

Part 9 에서 손글씨 숫자를 **64 픽셀을 한 줄로 펴서** DNN에 넣었습니다. 하지만 이런 방식에는 한계가 있습니다.

| 한계 | 설명 |
|------|------|
| 위치 정보 소실 | 같은 "3"이 조금만 이동해도 픽셀 순서가 다르면 다른 입력으로 취급 |
| 파라미터 폭발 | 실제 28x28 MNIST 라면 입력 784 → 은닉 512 만 해도 40만 파라미터, 이미지가 커질수록 감당 불가 |
| 지역 패턴 학습 실패 | "눈·코·입" 같이 **근처 픽셀의 패턴**을 학습하는 데 비효율 |

> 이를 해결하는 것이 **합성곱 신경망(Convolutional Neural Network, CNN)** 입니다.

### 다음 시간 예고

- **CNN(합성곱 신경망)**: 합성곱(Convolution)과 풀링(Pooling)
- **이미지 특성맵(Feature Map)** 의 이해
- **CNN의 대표 구조**: LeNet, VGG 스타일
- **이미지 분류 실습**: 진짜 MNIST / CIFAR-10